# 06 · B5 — What does each model cost to run?
Trains every model at the best L and measures, **on CPU** (a controller host is usually CPU-only): parameters (tree nodes for RF/XGBoost), model size, FLOPs per window, median and 95th-percentile latency at batch 1, throughput at batch 1024, and training time.

Close other heavy programs before running. Record the CPU model (printed below) in the paper.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 60)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
enable_determinism()
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench.experiments import run_grid, load_results, choose_best_L
DATASETS = active_datasets(cfg, 'B5')
best_L = choose_best_L(cfg)
print(json.load(open(os.path.join(cfg['out_dir'], 'environment.json'))).get('cpu_model'))

In [ ]:
run_grid(cfg, 'B5', DATASETS, ['multi'], ['SC'], [best_L], mp['sweep_seeds'][:3], cfg['models'], with_cost=True)

In [ ]:
res = load_results(cfg, DATASETS)
display(res[res.exp=='B5'].groupby(['dataset','model'])[['macro_f1','params','size_mb','flops_per_flow','latency_ms_b1','throughput_fps','train_time_s']].mean().round(4))